# FAIR Principles Analysis of a Sample Dataset

This notebook is a **step-by-step classroom demo** for Day 1.

It shows how to:

1. inspect a dataset,
2. review its metadata,
3. evaluate it against the **FAIR principles**,
4. identify **weaknesses and practical issues**,
5. summarize how well the dataset supports:
   - **Findable**
   - **Accessible**
   - **Interoperable**
   - **Reusable**

The goal is **not** to build a perfect formal evaluator.  
The goal is to help students **understand FAIR in practice**.

## Step 1 — Import libraries

We use only simple Python tools:

- `pandas` for tabular data
- `textwrap` for readable output

No internet connection is required.

In [1]:
import pandas as pd
from textwrap import fill

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 120)

## Step 2 — Create a small example dataset

We use a small fictional dataset called:

**Urban Air Quality Sample — 2025**

This dataset is intentionally **imperfect** so that we can discuss FAIR weaknesses clearly.

In [2]:
data = [
    {"city": "Karlsruhe", "date": "2025-01-01", "pm25": 18.2, "pm10": 27.4, "unit": "ug/m3", "source": "sensor_A"},
    {"city": "Heidelberg", "date": "2025-01-01", "pm25": 15.7, "pm10": 24.1, "unit": "ug/m3", "source": "sensor_B"},
    {"city": "Mannheim", "date": "2025-01-01", "pm25": None, "pm10": 30.2, "unit": "ug/m3", "source": "sensor_C"},
    {"city": "Karlsruhe", "date": "2025-01-02", "pm25": 19.4, "pm10": 28.0, "unit": "ug/m3", "source": "sensor_A"},
    {"city": "Heidelberg", "date": "2025-01-02", "pm25": 16.1, "pm10": None, "unit": "ug/m3", "source": "sensor_B"},
    {"city": "Karlsruhe", "date": "2025-01-02", "pm25": 19.4, "pm10": 28.0, "unit": "ug/m3", "source": "sensor_A"},  # duplicate row
]

df = pd.DataFrame(data)
df

,city,date,pm25,pm10,unit,source
0,Karlsruhe,2025-01-01,18.2,27.4,ug/m3,sensor_A
1,Heidelberg,2025-01-01,15.7,24.1,ug/m3,sensor_B
2,Mannheim,2025-01-01,NaN,30.2,ug/m3,sensor_C
3,Karlsruhe,2025-01-02,19.4,28.0,ug/m3,sensor_A
4,Heidelberg,2025-01-02,16.1,NaN,ug/m3,sensor_B
5,Karlsruhe,2025-01-02,19.4,28.0,ug/m3,sensor_A


## Step 3 — Look at the accompanying metadata

In a FAIR review, we do **not** look only at the raw data table.  
We also inspect the **metadata** and the dataset description.

Below, the metadata is again intentionally incomplete.

In [3]:
metadata = {
    "title": "Air quality data",
    "identifier": None,  # no DOI / no persistent identifier
    "repository": "Local course folder only",
    "authors": ["Course Demo Team"],
    "description": "Measurements collected from several cities in 2025.",
    "keywords": [],
    "license": None,  # missing license
    "access_url": None,  # no stable landing page
    "download_url": "shared_drive/final_version.csv",  # weak, not persistent
    "file_format": "CSV",
    "open_format": True,
    "standard_vocabulary": False,
    "schema_defined": False,
    "provenance_documented": False,
    "version": None,
    "contact_email": None,
    "readme_available": False,
    "units_documented": False,
    "temporal_coverage": "2025-01-01 to 2025-01-02",
    "spatial_coverage": None,
    "access_conditions": "Ask instructor",
}

meta_df = pd.DataFrame({
    "field": list(metadata.keys()),
    "value": [str(v) for v in metadata.values()]
})

meta_df

,field,value
0,title,Air quality data
1,identifier,None
2,repository,Local course folder only
3,authors,['Course Demo Team']
4,description,Measurements collected from several cities in 2025.
5,keywords,[]
6,license,None
7,access_url,None
8,download_url,shared_drive/final_version.csv
9,file_format,CSV


## Step 4 — Basic data quality inspection

Before evaluating FAIR, it helps to inspect obvious data issues:

- missing values,
- duplicate rows,
- unclear units,
- unclear structure,
- lack of documentation.

These issues strongly affect **Reusability** and sometimes **Interoperability**.

In [4]:
print("Shape:", df.shape)
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nColumn names:")
print(list(df.columns))

Shape: (6, 6)

Missing values per column:


,missing_values
city,0
date,0
pm25,1
pm10,1
unit,0
source,0



Duplicate rows: 1

Column names:
['city', 'date', 'pm25', 'pm10', 'unit', 'source']


## Step 5 — Define a simple FAIR review framework

This is a **teaching-oriented FAIR checklist**, not an official formal standard.

For each FAIR principle, we ask a few practical questions and assign:
- **1** = criterion satisfied
- **0** = criterion not satisfied

Then we summarize strengths and weaknesses.

In [5]:
fair_checks = {
    "Findable": [
        ("Persistent identifier exists", metadata["identifier"] is not None),
        ("Clear title provided", bool(metadata["title"])),
        ("Rich description available", bool(metadata["description"]) and len(metadata["description"]) > 30),
        ("Keywords provided", len(metadata["keywords"]) > 0),
        ("Stored in a proper repository", metadata["repository"] not in [None, "", "Local course folder only"]),
    ],
    "Accessible": [
        ("Access or landing page exists", metadata["access_url"] is not None),
        ("Download link exists", metadata["download_url"] is not None),
        ("Access conditions are clear", metadata["access_conditions"] not in [None, ""]),
        ("Contact information is available", metadata["contact_email"] is not None),
    ],
    "Interoperable": [
        ("Open file format is used", metadata["open_format"] is True),
        ("Schema or structure is documented", metadata["schema_defined"] is True),
        ("Standard vocabulary is used", metadata["standard_vocabulary"] is True),
        ("Units are documented", metadata["units_documented"] is True),
    ],
    "Reusable": [
        ("License is provided", metadata["license"] is not None),
        ("README / documentation exists", metadata["readme_available"] is True),
        ("Provenance is documented", metadata["provenance_documented"] is True),
        ("Version information is available", metadata["version"] is not None),
        ("Data quality issues are limited", df.isna().sum().sum() == 0 and df.duplicated().sum() == 0),
    ],
}

records = []
for principle, checks in fair_checks.items():
    for criterion, status in checks:
        records.append({
            "principle": principle,
            "criterion": criterion,
            "status": "Yes" if status else "No",
            "score": int(status)
        })

fair_df = pd.DataFrame(records)
fair_df

,principle,criterion,status,score
0,Findable,Persistent identifier exists,No,0
1,Findable,Clear title provided,Yes,1
2,Findable,Rich description available,Yes,1
3,Findable,Keywords provided,No,0
4,Findable,Stored in a proper repository,No,0
5,Accessible,Access or landing page exists,No,0
6,Accessible,Download link exists,Yes,1
7,Accessible,Access conditions are clear,Yes,1
8,Accessible,Contact information is available,No,0
9,Interoperable,Open file format is used,Yes,1


## Step 6 — Overall score by FAIR principle

This gives a quick overview of how well the dataset performs in each area.

In [6]:
summary = (
    fair_df.groupby("principle")
    .agg(
        passed=("score", "sum"),
        total=("score", "count")
    )
    .reset_index()
)

summary["score_percent"] = (100 * summary["passed"] / summary["total"]).round(1)
summary

,principle,passed,total,score_percent
0,Accessible,2,4,50.0
1,Findable,2,5,40.0
2,Interoperable,1,4,25.0
3,Reusable,0,5,0.0


## Step 7 — Investigate **Findable**

### What we check
A dataset is easier to **find** when it has:
- a persistent identifier such as a **DOI**,
- a clear title,
- searchable metadata,
- keywords,
- a proper repository landing page.

### Why this dataset is weak in Findability
We already see some problems:
- no DOI or persistent identifier,
- no keywords,
- it is stored only in a local folder,
- the title is too generic.

In [7]:
findable = fair_df[fair_df["principle"] == "Findable"].copy()
findable

,principle,criterion,status,score
0,Findable,Persistent identifier exists,No,0
1,Findable,Clear title provided,Yes,1
2,Findable,Rich description available,Yes,1
3,Findable,Keywords provided,No,0
4,Findable,Stored in a proper repository,No,0


In [8]:
findable_issues = [
    "No persistent identifier (for example DOI).",
    "Title is too generic: 'Air quality data' does not clearly describe scope or time range.",
    "No keywords are provided, so search and discovery are weaker.",
    "Dataset is not deposited in a recognized repository such as Zenodo or Figshare.",
]

findable_strengths = [
    "A title exists.",
    "A short description exists."
]

print("Strengths:")
for s in findable_strengths:
    print("-", s)

print("\nWeaknesses / issues:")
for w in findable_issues:
    print("-", w)

Strengths:
- A title exists.
- A short description exists.

Weaknesses / issues:
- No persistent identifier (for example DOI).
- Title is too generic: 'Air quality data' does not clearly describe scope or time range.
- No keywords are provided, so search and discovery are weaker.
- Dataset is not deposited in a recognized repository such as Zenodo or Figshare.


## Step 8 — Investigate **Accessible**

### What we check
A dataset is more **accessible** when users can:
- locate a landing page,
- understand how to access the data,
- download it reliably,
- contact someone if needed.

### Why this dataset is weak in Accessibility
The dataset has a weak download path and no stable landing page.
It also has no contact email.

In [9]:
accessible = fair_df[fair_df["principle"] == "Accessible"].copy()
accessible

,principle,criterion,status,score
5,Accessible,Access or landing page exists,No,0
6,Accessible,Download link exists,Yes,1
7,Accessible,Access conditions are clear,Yes,1
8,Accessible,Contact information is available,No,0


In [10]:
accessible_issues = [
    "No stable access URL or landing page is provided.",
    "The download path is not persistent: 'shared_drive/final_version.csv'.",
    "Access conditions are vague: 'Ask instructor' is not a clear long-term access policy.",
    "No contact email is provided."
]

accessible_strengths = [
    "There is at least a stated way to get the file.",
    "Access conditions are mentioned, even if they are weak."
]

print("Strengths:")
for s in accessible_strengths:
    print("-", s)

print("\nWeaknesses / issues:")
for w in accessible_issues:
    print("-", w)

Strengths:
- There is at least a stated way to get the file.
- Access conditions are mentioned, even if they are weak.

Weaknesses / issues:
- No stable access URL or landing page is provided.
- The download path is not persistent: 'shared_drive/final_version.csv'.
- Access conditions are vague: 'Ask instructor' is not a clear long-term access policy.
- No contact email is provided.


## Step 9 — Investigate **Interoperable**

### What we check
A dataset is more **interoperable** when it:
- uses open formats,
- has documented structure,
- uses standard vocabularies or conventions,
- clearly defines units and fields.

### Why this dataset is mixed in Interoperability
The file is CSV, which is good.  
But the structure and semantics are still weak.

In [11]:
interoperable = fair_df[fair_df["principle"] == "Interoperable"].copy()
interoperable

,principle,criterion,status,score
9,Interoperable,Open file format is used,Yes,1
10,Interoperable,Schema or structure is documented,No,0
11,Interoperable,Standard vocabulary is used,No,0
12,Interoperable,Units are documented,No,0


In [12]:
interoperable_issues = [
    "No formal schema is documented.",
    "No standard vocabulary or ontology is referenced.",
    "Units are not documented in the metadata, even though a 'unit' column exists.",
    "Column meaning and expected value conventions are not formally explained."
]

interoperable_strengths = [
    "The dataset uses an open format (CSV)."
]

print("Strengths:")
for s in interoperable_strengths:
    print("-", s)

print("\nWeaknesses / issues:")
for w in interoperable_issues:
    print("-", w)

Strengths:
- The dataset uses an open format (CSV).

Weaknesses / issues:
- No formal schema is documented.
- No standard vocabulary or ontology is referenced.
- Units are not documented in the metadata, even though a 'unit' column exists.
- Column meaning and expected value conventions are not formally explained.


## Step 10 — Investigate **Reusable**

### What we check
A dataset is more **reusable** when it includes:
- a clear license,
- documentation / README,
- provenance,
- version information,
- good data quality.

### Why this dataset is weak in Reusability
This dataset has several reuse problems:
- no license,
- no README,
- no provenance record,
- no version,
- missing values and a duplicate row.

In [13]:
reusable = fair_df[fair_df["principle"] == "Reusable"].copy()
reusable

,principle,criterion,status,score
13,Reusable,License is provided,No,0
14,Reusable,README / documentation exists,No,0
15,Reusable,Provenance is documented,No,0
16,Reusable,Version information is available,No,0
17,Reusable,Data quality issues are limited,No,0


In [14]:
reusable_issues = [
    "No usage license is provided, so reuse permissions are unclear.",
    "No README or user guide is available.",
    "No provenance is documented: we do not know exactly how the data was collected and processed.",
    "No version number is provided.",
    "The table contains missing values.",
    "The table contains a duplicate row."
]

reusable_strengths = [
    "The dataset has a short textual description.",
    "The dataset is small and readable, which helps manual inspection."
]

print("Strengths:")
for s in reusable_strengths:
    print("-", s)

print("\nWeaknesses / issues:")
for w in reusable_issues:
    print("-", w)

Strengths:
- The dataset has a short textual description.
- The dataset is small and readable, which helps manual inspection.

Weaknesses / issues:
- No usage license is provided, so reuse permissions are unclear.
- No README or user guide is available.
- No provenance is documented: we do not know exactly how the data was collected and processed.
- No version number is provided.
- The table contains missing values.
- The table contains a duplicate row.


## Step 11 — Final FAIR summary table

Now we combine the main issues into one easy classroom summary.

In [15]:
final_summary = pd.DataFrame([
    {
        "FAIR Principle": "Findable",
        "Strengths": "Has a title and short description.",
        "Weaknesses": "No DOI, no keywords, generic title, not in a proper repository.",
        "Teaching conclusion": "Partially findable, but weak in discovery support."
    },
    {
        "FAIR Principle": "Accessible",
        "Strengths": "A possible file path is given and access is mentioned.",
        "Weaknesses": "No landing page, weak download path, no contact information.",
        "Teaching conclusion": "Access is unclear and not robust."
    },
    {
        "FAIR Principle": "Interoperable",
        "Strengths": "Uses CSV, an open format.",
        "Weaknesses": "No schema, no standard vocabulary, units not documented.",
        "Teaching conclusion": "Technically open, but semantically weak."
    },
    {
        "FAIR Principle": "Reusable",
        "Strengths": "Human-readable sample with short description.",
        "Weaknesses": "No license, no README, no provenance, missing values, duplicate row.",
        "Teaching conclusion": "Reusability is weak and would need major improvement."
    },
])

final_summary

,FAIR Principle,Strengths,Weaknesses,Teaching conclusion
0,Findable,Has a title and short description.,"No DOI, no keywords, generic title, not in a proper repository.","Partially findable, but weak in discovery support."
1,Accessible,A possible file path is given and access is mentioned.,"No landing page, weak download path, no contact information.",Access is unclear and not robust.
2,Interoperable,"Uses CSV, an open format.","No schema, no standard vocabulary, units not documented.","Technically open, but semantically weak."
3,Reusable,Human-readable sample with short description.,"No license, no README, no provenance, missing values, duplicate row.",Reusability is weak and would need major improvement.


## Step 12 — What should be improved?

To improve the dataset, students should propose actions such as:

1. deposit the dataset in a trusted repository,
2. assign a **DOI**,
3. improve the title and add keywords,
4. add a stable landing page and contact email,
5. add a README and a clear license,
6. document units, schema, and provenance,
7. clean missing values and duplicates,
8. add version information.

These are exactly the kinds of improvements that make FAIR principles practical.

In [16]:
improvements = pd.DataFrame({
    "improvement_action": [
        "Assign a DOI or persistent identifier",
        "Deposit dataset in Zenodo / Figshare / institutional repository",
        "Add keywords and improve the title",
        "Provide stable landing and download links",
        "Add contact email",
        "Add README and clear license",
        "Document schema, fields, and units",
        "Use standard vocabulary where possible",
        "Document provenance and version",
        "Clean missing values and duplicate rows",
    ]
})

improvements

,improvement_action
0,Assign a DOI or persistent identifier
1,Deposit dataset in Zenodo / Figshare / institutional repository
2,Add keywords and improve the title
3,Provide stable landing and download links
4,Add contact email
5,Add README and clear license
6,"Document schema, fields, and units"
7,Use standard vocabulary where possible
8,Document provenance and version
9,Clean missing values and duplicate rows


## Step 13 — Short classroom interpretation

### Suggested conclusion
This dataset is a good teaching example because it is:

- **partially FAIR**, not completely FAIR,
- easy to inspect,
- rich enough to show typical weaknesses,
- realistic for classroom discussion.

Students can now:
- evaluate another dataset,
- compare strengths and weaknesses,
- suggest FAIR improvements.